<a href="https://colab.research.google.com/github/Vandana-2305/GenAI-Project/blob/main/GenAI_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

# 1. Configuration & Constants
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "./results"
SAVE_DIR = "./fine_tuned_distilbert_imdb"
MAX_LENGTH = 128
NUM_LABELS = 2


# 2. Compute Evaluation Metrics
def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(pred.label_ids, preds)
    return {"accuracy": acc}


def main():
    # 3. Load & Subset Dataset (Updated repo ID)
    dataset = load_dataset("stanfordnlp/imdb")

    small_train = (
        dataset["train"].shuffle(seed=42).select(range(2000))
    )
    small_test = (
        dataset["test"].shuffle(seed=42).select(range(500))
    )

    # 4. Tokenization
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_function(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
        )

    train_ds = small_train.map(tokenize_function, batched=True)
    test_ds = small_test.map(tokenize_function, batched=True)

    # 5. Load Model
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    )

    # 6. Training Setup
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=2,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        eval_strategy="epoch",
        logging_steps=50,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics,
    )

    # 7. Train, Evaluate & Save
    trainer.train()

    metrics = trainer.evaluate()
    print("\n--- Evaluation Metrics ---")
    for key, value in metrics.items():
        print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

    model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)


if __name__ == "__main__":
    main()

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Epoch,Training Loss,Validation Loss,Accuracy
1,0.380451,0.463636,0.810000
2,0.268494,0.581856,0.824000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.268494,0.581856,2,0.824000



--- Evaluation Metrics ---
eval_loss: 0.5819
eval_accuracy: 0.8240


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]